In [2]:
from glob import glob
import pandas as pd
import numpy as np


# Constants/parameters

In [3]:
# Marginal cost below includes fixed cost
SCREENING_SURVEY_FIXED_COST = 0

SATELLITE_IMAGERY_COST = 0

# From Table 1, 
# https://openknowledge.worldbank.org/server/api/core/bitstreams/2a1dd421-d0ae-5d6f-9c15-209565e9edd8/content
LSMS_PER_HH_COST_2014_PRICES = 322.99

POLICY_COST_CURRENCY_YEAR = 2017

POVERTY_RATE_TARGET = 0.01

# Setup

In [4]:
def interpolate_policy_cost(df, post_transfer_poverty_rate):
    df_sorted = (
        df.sort_values('post_transfer_poverty_rate').drop_duplicates('post_transfer_poverty_rate')
    )
    return float(np.interp(
        post_transfer_poverty_rate,
        df_sorted['post_transfer_poverty_rate'],
        df_sorted['policy_cost_per_capita'],
    ))

In [5]:
aux_files = glob('/data/eop/compiled_country_data/auxiliary_data/auxiliary_data_*.csv')
latest_file = max(aux_files, key=lambda x: x.split('_')[-1].split('.')[0])
aux_data = pd.read_csv(latest_file)
togo_aux_data = aux_data[aux_data.country_code == 'TGO']

secondary_aux_files = glob('/data/eop/compiled_country_data/auxiliary_data/secondary_auxiliary_data*.csv')
latest_file = max(secondary_aux_files, key=lambda x: x.split('_')[-1].split('.')[0])
secondary_aux_data = pd.read_csv(latest_file)

policy_cost_inflation_adjustment = (
    1
    / secondary_aux_data[
        secondary_aux_data["indicator"]
        == f"conversion_factor_nominal_USD_2023_to_{POLICY_COST_CURRENCY_YEAR}"
    ]["value"]
    .values[0]
    .item()
)

policy_cost_conversion_factor_to_real_usd_2023 = (
    togo_aux_data["total_population_survey_year"].values[0]
    * 365 # days per year
    * policy_cost_inflation_adjustment # from 2017 to 2023 nominal
    *  (
        togo_aux_data[f"PPP_conversion_factor_{POLICY_COST_CURRENCY_YEAR}"].values[0]
        / togo_aux_data[f"market_exchange_rate_{POLICY_COST_CURRENCY_YEAR}"].values[0]
    )  # from PPP to nominal USD in year
)

In [6]:
# LSMS costs
lsms_costs_inflation_adjustment = (
    1
    / secondary_aux_data[
        secondary_aux_data["indicator"]
        == "conversion_factor_nominal_USD_2023_to_2014"
    ]["value"]
    .values[0]
    .item()
)
lsms_per_hh_cost_2023_prices = LSMS_PER_HH_COST_2014_PRICES * lsms_costs_inflation_adjustment
togo_survey = pd.read_parquet('/data/eop/country_data/TGO/cleaned/full.parquet')
lsms_cost_real_2023 = lsms_per_hh_cost_2023_prices * len(togo_survey)

# Estimating survey d=20 cost

In [7]:
survey_d20_simulation_results = pd.read_csv(
    '/data/eop/simulation_results/TGO_20260514/TGO/year=2017_d=20/output_gt_continuous_gap.csv'
)
survey_d20_policy_cost = interpolate_policy_cost(
    survey_d20_simulation_results, POVERTY_RATE_TARGET
)
survey_d20_policy_cost_real_2023 = (
    survey_d20_policy_cost * policy_cost_conversion_factor_to_real_usd_2023
)

In [8]:
# Table 8 here: https://openknowledge.worldbank.org/server/api/core/bitstreams/79e09246-64db-5060-b959-10a417aaeb74/content
bfa_cost_per_hh = 5.69
chad_cost_per_hh = 9.50
niger_cost_per_hh = 6.80
mali_cost_per_hh = 4

costs_per_hh = [bfa_cost_per_hh, chad_cost_per_hh, niger_cost_per_hh, mali_cost_per_hh]

median_pmt_cost_per_hh = np.median(costs_per_hh)

# Table 2 here: https://openknowledge.worldbank.org/server/api/core/bitstreams/79e09246-64db-5060-b959-10a417aaeb74/content
# bfa, niger, mali all 2016. Chad unspecified.
assumed_screening_cost_currency_year = 2016

screening_costs_inflation_adjustment = (
    1
    / secondary_aux_data[
        secondary_aux_data["indicator"]
        == f"conversion_factor_nominal_USD_2023_to_{assumed_screening_cost_currency_year}"
    ]["value"]
    .values[0]
    .item()
)

weighted_average_household_size = (
    (togo_survey.hh_size * togo_survey.hh_wgt).sum() / togo_survey.hh_wgt.sum()
)
estimated_household_count_survey_year = (
    togo_aux_data["total_population_survey_year"].values[0] / weighted_average_household_size
)

screening_per_hh_cost_real_2023 = median_pmt_cost_per_hh * screening_costs_inflation_adjustment

screening_cost_real_2023 = (
    estimated_household_count_survey_year * screening_per_hh_cost_real_2023
    + SCREENING_SURVEY_FIXED_COST * screening_costs_inflation_adjustment
)

In [9]:
survey_d20_cost_real_2023 = survey_d20_policy_cost_real_2023 + screening_cost_real_2023 + lsms_cost_real_2023

In [10]:
survey_d20_cost_real_2023 / 1e9

1.445147798312961

In [11]:
(survey_d20_cost_real_2023 - survey_d20_policy_cost_real_2023) / survey_d20_cost_real_2023

0.012530612329561224

# Estimating satellite imagery cost

In [12]:
alpha_earth_simulation_results = pd.read_csv(
    '/data/eop/simulation_results/TGO_20260514/TGO_alpha_earth/geo_extrapolation/year=2017/output_gt_continuous_gap.csv'
)
alpha_earth_policy_cost = interpolate_policy_cost(alpha_earth_simulation_results, 0.01)
alpha_earth_policy_cost_real_2023 = (
    alpha_earth_policy_cost * policy_cost_conversion_factor_to_real_usd_2023
)

In [13]:
satellite_imagery_cost = estimated_household_count_survey_year * SATELLITE_IMAGERY_COST 
satellite_imagery_cost_real_2023 = (
    satellite_imagery_cost * policy_cost_conversion_factor_to_real_usd_2023
)

In [14]:
alpha_earth_cost_real_2023 = (
    alpha_earth_policy_cost_real_2023 + satellite_imagery_cost_real_2023 + lsms_cost_real_2023
)

In [15]:
alpha_earth_cost_real_2023 / 1e9

1.7793223452213638

# Create macros

In [ ]:
survey_non_transfer_costs = (survey_d20_cost_real_2023 - survey_d20_policy_cost_real_2023)
survey_non_transfer_fraction_of_transfer = survey_non_transfer_costs / survey_d20_policy_cost_real_2023
survey_non_transfer_fraction_of_total = survey_non_transfer_costs / survey_d20_cost_real_2023
alpha_earth_cost_increase_over_survey = (alpha_earth_cost_real_2023/survey_d20_cost_real_2023 - 1) 

alpha_earth_non_transfer_costs = (alpha_earth_cost_real_2023 - alpha_earth_policy_cost_real_2023)
macros = {
    "surveyTotalCostBillion": f"{survey_d20_cost_real_2023 / 1e9:.2f}",
    "alphaEarthTotalCostBillion": f"{alpha_earth_cost_real_2023 / 1e9:.2f}",
    "alphaEarthCostIncreaseOverSurvey": f"{alpha_earth_cost_increase_over_survey * 100:.0f}",
    "lsmsPerHhCost": f"{lsms_per_hh_cost_2023_prices:.0f}",
    "screeningPerHhCost": f"{screening_per_hh_cost_real_2023:.2f}",
    "surveyNonTransferCostMillion": f"{survey_non_transfer_costs / 1e6:.0f}",
    "surveyNonTransferPercentageOfTransfer": f"{survey_non_transfer_fraction_of_transfer * 100:.1f}",
    "surveyNonTransferPercentageOfTotal": f"{survey_non_transfer_fraction_of_total * 100:.1f}",
    "alphaEarthNonTransferCostMillion": f"{alpha_earth_non_transfer_costs / 1e6:.0f}",
}

lines = [f"\\newcommand{{\\{name}}}{{{value}}}" for name, value in macros.items()]
output = "\n".join(lines)

output_path = '/home/selker/eop/eop/togo_cost_exercise/togo_costing_macros.tex'
with open(output_path, 'w') as f:
    f.write(output + '\n')

print(output)


\newcommand{\surveyTotalCostBillion}{1.45}
\newcommand{\alphaEarthTotalCostBillion}{1.78}
\newcommand{\alphaEarthCostIncreaseOverSurvey}{23}
\newcommand{\lsmsPerHhCost}{416}
\newcommand{\screeningPerHhCost}{7.93}
\newcommand{\surveyNonTransferCostMillion}{18}
\newcommand{\surveyNonTransferPercentageOfTransfer}{1.3}
\newcommand{\surveyNonTransferPercentageOfTotal}{1.3}
\newcommand{\alphaEarthNonTransferCostMillion}{3}


In [27]:

def fmt_m(val):
    return f"\\${val/1e6:,.0f}"

def fmt_m_bold(val):
    return r"\textbf{" + fmt_m(val) + "}"

survey_admin = survey_d20_cost_real_2023 - survey_d20_policy_cost_real_2023
alpha_earth_admin = alpha_earth_cost_real_2023 - alpha_earth_policy_cost_real_2023

latex = r"""\begin{tabular}{lrr}
\toprule
 & \textbf{Survey} & \textbf{Satellite} \\
\midrule
\textbf{Total program cost} & """ + fmt_m_bold(survey_d20_cost_real_2023) + r" & " + fmt_m_bold(alpha_earth_cost_real_2023) + r" \\" + """
\\quad Cost of transfers & """ + fmt_m(survey_d20_policy_cost_real_2023) + r" & " + fmt_m(alpha_earth_policy_cost_real_2023) + r" \\" + """
\\quad \\textbf{Administrative costs} & """ + fmt_m_bold(survey_admin) + r" & " + fmt_m_bold(alpha_earth_admin) + r" \\" + """
\\qquad Screening & """ + fmt_m(screening_cost_real_2023) + r" & " + fmt_m(satellite_imagery_cost_real_2023) + r" \\" + """
\\qquad Training data & """ + fmt_m(lsms_cost_real_2023) + r" & " + fmt_m(lsms_cost_real_2023) + r""" \\
\bottomrule
\end{tabular}"""

output_path = '/home/selker/eop/eop/togo_cost_exercise/appendix-table-togo-costs.tex'
with open(output_path, 'w') as f:
    f.write(latex + '\n')

print(latex)


\begin{tabular}{lrr}
\toprule
 & \textbf{Survey} & \textbf{Satellite} \\
\midrule
\textbf{Total program cost} & \textbf{\$1,445} & \textbf{\$1,779} \\
\quad Cost of transfers & \$1,427 & \$1,777 \\
\quad \textbf{Administrative costs} & \textbf{\$18} & \textbf{\$3} \\
\qquad Screening & \$16 & \$0 \\
\qquad Training data & \$3 & \$3 \\
\bottomrule
\end{tabular}
